# 02 – Enriquecimento externo e bases analíticas

A Gold traz apenas os resultados educacionais (avaliação e metas). Para explicar por que alguns
municípios alfabetizam mais que outros, o projeto agrega fontes públicas com o contexto territorial e
socioeconômico de cada município. Em seguida, monta as bases que vão alimentar os modelos.

**Objetivos deste notebook**
1. Carregar as fontes externas (IBGE e Atlas do Desenvolvimento Humano) e consolidá-las em uma tabela por município.
2. Verificar a cobertura: quantos municípios avaliados encontram correspondência nessas fontes.
3. Montar as bases analíticas de alunos e de municípios, seguindo o princípio anti-leakage.
4. Salvar tudo em Parquet (`data/processed/`) e checar os valores faltantes.

**Fontes externas**

| Fonte | Conteúdo | Granularidade | Período |
|---|---|---|---|
| IBGE – Relação de municípios | coordenadas, capital, UF | município | atual |
| IBGE – Censos e estimativas populacionais | população total | município × ano | 2010–2025 |
| Atlas do Desenvolvimento Humano (PNUD/IPEA/FJP) | IDHM e 23 indicadores de renda, educação, saúde e infraestrutura | município | Censo 2010 |

**Onde está o código**
- `src/preprocessing/external.py`: download das fontes (`baixar_fontes_externas()`) e montagem da tabela municipal (`construir_features_municipio()`).
- `src/preprocessing/features.py`: construção e gravação das bases analíticas (`salvar_bases()`) e lista de variáveis excluídas por risco de leakage.

Os arquivos externos ficam versionados em `data/external/`.

> **Limitação conhecida:** os indicadores socioeconômicos detalhados são do Censo 2010, o último Atlas
> municipal completo. Eles entram como **proxy estrutural** do município: pobreza, escolaridade dos adultos
> e infraestrutura mudam devagar e preservam a ordem relativa entre os municípios.

In [1]:
# Configura o caminho do projeto para importar os módulos de src/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

## 1. Fontes externas

In [2]:
# Baixa as fontes (somente se ainda não existirem em data/external) e monta a tabela municipal.
from src.preprocessing.external import baixar_fontes_externas, construir_features_municipio, ATLAS_COLS

baixar_fontes_externas()
ext = construir_features_municipio()
print(ext.shape)
ext.head()

(5571, 33)


,id_municipio,nome_municipio,latitude,longitude,capital,populacao_2022,crescimento_pop_2010_2022,idhm,idhm_educacao,idhm_longevidade,idhm_renda,renda_per_capita,gini,pct_extrema_pobreza,pct_pobres,pct_criancas_extrema_pobreza,taxa_analfabetismo_18mais,anos_estudo_esperados,taxa_freq_escola_4a5,taxa_freq_escola_5a6,taxa_atraso_2anos_fund,pct_criancas_lar_sem_fund_completo,pct_agua_encanada,pct_sem_agua_esgoto_adequado,pct_energia_eletrica,mortalidade_ate_5anos,esperanca_vida,razao_dependencia,grau_formalizacao_trabalho,taxa_desocupacao,log_populacao,taxa_urbanizacao_2010,porte_municipio
0,5200050,Abadia de Goiás,-16.75730,-49.4412,0,19128.0,1.781850,0.708,0.622,0.830,0.687,574.96,0.42,1.97,23.27,3.68,8.36,7.94,83.29,95.44,22.55,32.03,93.06,0.46,99.33,15.86,74.82,44.64,50.54,5.39,4.281670,0.738947,5–20 mil
1,3100104,Abadia dos Dourados,-18.48310,-47.3916,0,6272.0,-0.064439,0.689,0.563,0.839,0.693,596.18,0.47,1.85,27.40,2.83,9.67,8.68,30.26,74.76,20.39,35.53,88.50,1.26,98.55,17.22,75.35,42.76,41.97,4.09,3.797406,0.624851,5–20 mil
2,5200100,Abadiânia,-16.19700,-48.7057,0,17232.0,0.093609,0.689,0.579,0.841,0.671,519.87,0.43,2.00,31.19,2.89,11.67,9.31,53.97,75.62,14.92,34.35,94.50,3.82,99.68,14.84,75.46,44.72,39.37,3.65,4.236336,0.684013,5–20 mil
3,3100203,Abaeté,-19.15510,-45.4444,0,22675.0,-0.000661,0.698,0.556,0.848,0.720,707.24,0.54,1.61,26.65,3.76,9.28,9.19,67.89,95.99,15.83,45.20,98.40,0.17,99.92,16.27,75.90,45.23,55.37,5.34,4.355547,0.868400,20–100 mil
4,1500107,Abaetetuba,-1.72183,-48.8788,0,158188.0,0.121106,0.628,0.537,0.798,0.579,293.01,0.53,18.98,65.93,25.66,14.97,8.99,83.51,95.78,24.47,38.08,68.86,39.80,91.80,20.42,72.89,55.37,32.99,8.66,5.199174,0.588221,100–500 mil


In [3]:
# Dicionário dos indicadores do Atlas utilizados.
pd.DataFrame(list(ATLAS_COLS.items()), columns=["código Atlas", "nome no projeto"])

,código Atlas,nome no projeto
0,idhm,idhm
1,idhm_e,idhm_educacao
2,idhm_l,idhm_longevidade
3,idhm_r,idhm_renda
4,rdpc,renda_per_capita
5,gini,gini
6,pind,pct_extrema_pobreza
7,ppob,pct_pobres
8,pindcri,pct_criancas_extrema_pobreza
9,t_analf18m,taxa_analfabetismo_18mais


In [4]:
# Cobertura: quantos municípios da avaliação encontram correspondência nas fontes externas?
from src.preprocessing.gold import carregar_gold
gm = carregar_gold("gold_municipio_desempenho")
ids = set(gm["id_municipio"])
cobertura = {
    "IBGE (municípios)": len(ids & set(ext["id_municipio"])) / len(ids),
    "Atlas 2010 (IDHM)": len(ids & set(ext.dropna(subset=["idhm"])["id_municipio"])) / len(ids),
    "População 2022": len(ids & set(ext.dropna(subset=["populacao_2022"])["id_municipio"])) / len(ids),
}
pd.Series(cobertura, name="cobertura").map("{:.2%}".format)

IBGE (municípios)    100.00%
Atlas 2010 (IDHM)     99.91%
População 2022       100.00%
Name: cobertura, dtype: object

Os 5 municípios sem dados do Atlas 2010 (Balneário Rincão/SC, Mojuí dos Campos/PA, Paraíso das Águas/MS,
Pescaria Brava/SC e Pinto Bandeira/RS) foram instalados em 2013, depois do Censo. Seus valores ficam
nulos e são tratados pela imputação dentro da pipeline de modelagem.

## 2. Bases analíticas

### Princípio anti-leakage: só usar o que se conhece *antes* do resultado

| Base | Unidade | Alvo | Variáveis explicativas |
|---|---|---|---|
| `base_alunos_2024` | aluno com avaliação válida em 2024 | `alfabetizado` | rede, caderno, território, socioeconomia, metas e **resultados de 2023** do município e da UF |
| `base_municipios_2023` | município com meta 2024 | `nao_atingiu_meta` (2024) | indicadores de **2023** + socioeconomia + esforço exigido pela meta |
| `base_municipios_2024` | município | — (projeção) | indicadores de **2024** para estimar o risco de não atingir a meta de 2025 |

### Variáveis excluídas do modelo de alunos e o motivo:

In [5]:
from src.preprocessing.features import VARIAVEIS_PROIBIDAS, salvar_bases

(pd.DataFrame(list(VARIAVEIS_PROIBIDAS.items()), columns=["variável", "motivo da exclusão"])
   .style.hide(axis="index")
   .set_properties(**{"text-align": "left", "white-space": "normal"}))

variável,motivo da exclusão
proficiencia,É a nota que define o alvo (alfabetizado = nota ≥ 743). Usá-la entregaria a resposta ao modelo.
peso_aluno,Calculado pelo INEP depois da prova. Não está disponível no momento da previsão.
presenca,"Igual para todos os alunos da base, que só inclui presentes. Não diferencia ninguém."
preenchimento_caderno,"Critério usado para selecionar a base. Igual para todos, sem poder de diferenciação."
id_aluno,Código de identificação. Não traz nenhuma informação sobre o aluno.
id_escola,Faria o modelo memorizar escolas em vez de aprender padrões e não funcionaria para escolas novas.
taxa_alfabetizacao (mesmo ano),Já inclui o resultado do próprio aluno. O modelo usa a taxa do ano anterior.
nivel_alfabetizacao / percentual_participacao (mesmo ano),Vêm da mesma prova que gera o alvo. O modelo usa os valores do ano anterior.


In [6]:
# Gera e salva as bases analíticas em data/processed/.
bases = salvar_bases()
pd.DataFrame([(k, *v.shape) for k, v in bases.items()], columns=["base", "linhas", "colunas"])

,base,linhas,colunas
0,base_alunos_2024,49651,58
1,base_municipios_2023,5232,51
2,base_municipios_2024,5448,47
3,features_municipio_externas,5571,33


In [7]:
# Prévia da base de alunos.
alunos = bases["base_alunos_2024"]
print(f"Alunos: {len(alunos):,} | municípios: {alunos['id_municipio'].nunique():,} | "
      f"escolas: {alunos['id_escola'].nunique():,} | % alfabetizados: {alunos['alfabetizado'].mean():.1%}")
alunos.head()

Alunos: 49,651 | municípios: 4,745 | escolas: 25,538 | % alfabetizados: 59.8%


,id_aluno,id_escola,id_municipio,sigla_uf,regiao,rede_ensino,caderno,alfabetizado,media_portugues_estadual_ant,media_portugues_municipal_ant,media_portugues_publica_ant,taxa_alfabetizacao_estadual_ant,taxa_alfabetizacao_municipal_ant,taxa_alfabetizacao_publica_ant,participacao_ant,nivel_alfabetizacao_ant,taxa_uf_publica_ant,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,latitude,...,renda_per_capita,gini,pct_extrema_pobreza,pct_pobres,pct_criancas_extrema_pobreza,taxa_analfabetismo_18mais,anos_estudo_esperados,taxa_freq_escola_4a5,taxa_freq_escola_5a6,taxa_atraso_2anos_fund,pct_criancas_lar_sem_fund_completo,pct_agua_encanada,pct_sem_agua_esgoto_adequado,pct_energia_eletrica,mortalidade_ate_5anos,esperanca_vida,razao_dependencia,grau_formalizacao_trabalho,taxa_desocupacao,log_populacao,taxa_urbanizacao_2010,porte_municipio,taxa_rede_aluno_ant,media_rede_aluno_ant,esforco_meta_2024
0,22015252,60006989,2202208,PI,Nordeste,Municipal,1,1,NaN,742.7673,742.7673,NaN,52.24,52.24,92.74,2.0,52.48,56.83,61.30,65.60,69.65,73.42,76.87,80.0,-4.8217,...,401.74,0.53,9.49,53.87,14.66,23.48,9.69,93.32,96.29,19.25,39.94,91.53,3.79,97.91,22.21,72.58,47.65,34.67,8.64,4.660799,0.741993,20–100 mil,52.24,742.7673,4.59
1,23077065,60007337,2307809,CE,Nordeste,Municipal,1,1,NaN,807.9766,807.9766,NaN,94.18,94.18,94.90,5.0,84.46,80.00,80.00,80.00,80.00,80.00,80.00,80.0,-3.1285,...,266.87,0.54,20.27,71.84,28.84,30.95,9.28,92.10,97.74,12.16,44.76,83.32,11.74,98.25,31.60,69.05,62.81,26.39,7.30,4.411603,0.624823,20–100 mil,94.18,807.9766,-14.18
2,27025381,60013918,2707503,AL,Nordeste,Municipal,1,0,737.9699,756.3131,751.5900,46.86,67.71,62.34,94.35,3.0,43.88,69.69,71.61,73.44,75.20,76.88,78.48,80.0,-10.1849,...,195.26,0.61,38.12,79.28,48.69,37.12,8.49,86.08,92.44,26.78,56.97,71.29,6.91,99.65,30.54,70.46,58.54,25.92,13.96,4.302807,0.341678,20–100 mil,67.71,756.3131,1.98
3,29184649,60015303,2925956,BA,Nordeste,Municipal,1,1,NaN,733.9164,733.9164,NaN,48.41,48.41,92.83,1.0,36.78,53.58,58.68,63.59,68.24,72.55,76.48,80.0,-12.4053,...,226.94,0.50,25.44,70.55,35.24,32.56,7.71,91.08,95.31,27.61,56.29,58.66,31.17,93.96,26.20,71.06,62.22,17.17,7.48,4.293628,0.299467,5–20 mil,48.41,733.9164,5.17
4,51005065,60040461,5102504,MT,Centro-Oeste,Municipal,1,1,NaN,739.6200,739.6200,NaN,47.53,47.53,87.98,1.0,55.03,52.83,58.07,63.13,67.91,72.35,76.39,80.0,-16.0764,...,590.43,0.53,4.69,37.01,7.15,10.93,9.09,70.25,85.57,17.24,28.72,95.51,7.91,99.04,21.28,73.76,49.66,54.43,9.09,4.952700,0.870665,20–100 mil,47.53,739.6200,5.30


In [8]:
# Percentual de valores faltantes por variável (apenas as que têm algum faltante).
faltantes = alunos.isna().mean().sort_values(ascending=False)
faltantes[faltantes > 0].map("{:.2%}".format).to_frame("% faltante")

,% faltante
media_portugues_estadual_ant,58.82%
taxa_alfabetizacao_estadual_ant,58.82%
media_portugues_publica_ant,10.91%
taxa_alfabetizacao_publica_ant,10.91%
participacao_ant,4.71%
meta_alfabetizacao_2024,4.71%
nivel_alfabetizacao_ant,4.71%
esforco_meta_2024,4.71%
meta_alfabetizacao_2025,3.22%
meta_alfabetizacao_2026,3.22%


**Leitura dos faltantes**
- `taxa_alfabetizacao_estadual_ant` e `media_portugues_estadual_ant` (~59%): a maioria dos municípios não tem
  rede estadual nos anos iniciais. A ausência é informativa, não um erro. Por isso, na pipeline, a imputação
  numérica é acompanhada de indicadores de ausência (`add_indicator=True`).
- `taxa_alfabetizacao_publica_ant` e `media_portugues_publica_ant` (~11%): concentrados em São Paulo. A base do
  INEP de 2023 não traz o resultado da rede pública para 564 dos 643 municípios paulistas. Em 2024 o dado está completo.
- Metas e resultados de 2023 (~2–5%): municípios sem resultado divulgado em 2023.
- Valores próximos de 0% nas variáveis do Atlas: os 5 municípios instalados em 2013.

In [9]:
# Base municipal de treino (2023 -> meta 2024).
mun = bases["base_municipios_2023"]
print(f"Municípios: {len(mun):,} | % que NÃO atingiram a meta de 2024: {mun['nao_atingiu_meta'].mean():.1%}")
mun.head()

Municípios: 5,232 | % que NÃO atingiram a meta de 2024: 46.7%


,id_municipio,sigla_uf,regiao,taxa_atual,media_atual,participacao_atual,nivel_atual,taxa_estadual_atual,media_estadual_atual,taxa_uf_publica_atual,nome_municipio,latitude,longitude,capital,populacao_2022,crescimento_pop_2010_2022,idhm,idhm_educacao,idhm_longevidade,idhm_renda,renda_per_capita,gini,pct_extrema_pobreza,pct_pobres,pct_criancas_extrema_pobreza,...,anos_estudo_esperados,taxa_freq_escola_4a5,taxa_freq_escola_5a6,taxa_atraso_2anos_fund,pct_criancas_lar_sem_fund_completo,pct_agua_encanada,pct_sem_agua_esgoto_adequado,pct_energia_eletrica,mortalidade_ate_5anos,esperanca_vida,razao_dependencia,grau_formalizacao_trabalho,taxa_desocupacao,log_populacao,taxa_urbanizacao_2010,porte_municipio,meta_alvo,esforco_meta,distancia_meta_2030,gap_para_uf,taxa_proximo_ano,meta_ano,status_meta,nao_atingiu_meta,ano_referencia
0,1100015,RO,Norte,64.55,758.3304,89.37,3.0,NaN,NaN,64.6,Alta Floresta D'Oeste,-11.92830,-61.9953,0.0,21494.0,-0.118809,0.641,0.526,0.763,0.657,476.99,0.58,14.29,47.12,19.01,...,8.67,39.18,70.79,17.82,45.73,93.69,9.31,93.98,25.49,70.75,47.37,41.68,6.07,4.332317,0.572729,20–100 mil,67.08,2.53,15.45,-0.05,67.79,67.08,ATINGIDA,0,2023
1,1100023,RO,Norte,62.30,757.0999,89.79,3.0,NaN,NaN,64.6,Ariquemes,-9.90571,-63.0325,0.0,96833.0,0.071719,0.702,0.600,0.806,0.716,689.95,0.53,4.36,29.04,6.65,...,9.18,55.34,77.46,16.74,29.58,98.54,11.40,98.58,20.53,73.36,46.68,52.81,5.11,4.986023,0.846956,20–100 mil,65.22,2.92,17.70,-2.30,65.62,65.22,ATINGIDA,0,2023
2,1100031,RO,Norte,69.10,767.8763,90.48,3.0,NaN,NaN,64.6,Cabixi,-13.49450,-60.5520,0.0,5351.0,-0.152384,0.650,0.559,0.757,0.650,457.17,0.51,7.27,46.98,8.51,...,10.09,66.92,85.31,12.93,41.25,95.49,3.31,96.36,26.58,70.39,48.12,39.38,2.90,3.728435,0.426580,5–20 mil,70.85,1.75,10.90,4.50,75.88,70.85,ATINGIDA,0,2023
3,1100049,RO,Norte,62.51,755.5645,84.44,3.0,66.13,761.9918,64.6,Cacoal,-11.43430,-61.4562,0.0,86887.0,0.105798,0.718,0.620,0.821,0.727,738.06,0.57,5.97,29.82,8.77,...,9.74,62.41,85.61,12.58,25.72,97.96,1.85,98.89,15.38,74.27,43.79,55.07,6.28,4.938955,0.788060,20–100 mil,65.39,2.88,17.49,-2.09,65.81,65.39,ATINGIDA,0,2023
4,1100056,RO,Norte,58.53,746.4629,92.12,2.0,NaN,NaN,64.6,Cerejeiras,-13.18700,-60.8168,0.0,15890.0,-0.066886,0.692,0.602,0.799,0.688,577.18,0.50,4.72,34.85,6.93,...,10.22,61.27,84.97,10.27,35.81,97.53,5.34,98.85,19.42,72.94,45.67,45.54,5.72,4.201124,0.846732,5–20 mil,62.09,3.56,21.47,-6.07,66.81,62.09,ATINGIDA,0,2023


## Resumo

- As fontes externas cobrem 100% dos municípios avaliados (99,9% no Atlas, sem os 5 municípios criados em 2013).
- Foram geradas três bases analíticas em `data/processed/`:
  - **alunos 2024**: 49.651 alunos, 59,8% alfabetizados;
  - **municípios 2023**: 5.232 municípios, 46,7% não atingiram a meta de 2024;
  - **municípios 2024**: 5.448 municípios, base da projeção para 2025.

Próximo passo: notebook 03, com a análise exploratória dessas bases.